In [9]:
# Convert mat to nifti files
from glob import glob
from cmcmultimodal.core.io import mat2nii
from pathlib import Path
import os

modality = ''
subject = 'Vlad' #'Moe' #'Vlad'

my_files = glob(f'/Users/Vasilis/CMC_data/{subject}/PSOCT/{modality}/Slice*.mat')
seq_params = f'/Users/Vasilis/CMC_data/{subject}/PSOCT/seq_params.json'
highres_files = []
lowres_files = []
for f in my_files:
    if os.path.exists(f.replace('.mat','.nii.gz')):
        continue
    nii_highres, nii_lowres = mat2nii(f, seq_params,
                                      nii_lowres_file=os.path.join(Path(f).parent,'lowres/', Path(f).name.replace('.mat','.nii.gz')),
                                      downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)


In [ ]:
# Zero pad the slides to the maximum shape
from pathlib import Path
from cmcmultimodal.core.io import pad_all_slides

modality = ''

inp_path = Path('/Users/Vasilis/CMC_data/Vlad/PSOCT') / modality / 'lowres'
pad_all_slides(inp_path, inp_path)
inp_path = Path('/Users/Vasilis/CMC_data/Vlad/PSOCT') / modality
pad_all_slides(inp_path, inp_path)

New slide shape for zero-padding: (1270, 820, 1)
New slide shape for zero-padding: (12700, 8200, 1)


In [ ]:
# TODO delete this section after all data have been flipped and analysis worked correctly
# Only to be used once!!! - This is for Moe
import subprocess
import glob
from pathlib import Path

inp_path = Path('/Users/Vasilis/CMC_data/Moe/PSOCT')
modalities = []#['Cross', 'Retardance', 'Orientation', 'Reflectivity']

for mod in modalities:
    files = (inp_path / mod).glob('Slice_*_En*.nii.gz')
    for f in files:
        input  = f
        output = f

        subprocess.run(["fslswapdim",
                        input,
                        "-x", "y", "z",
                        output])
    
    files = (inp_path / mod / 'lowres').glob('Slice_*_En*.nii.gz')
    for f in files:
        input  = f
        output = f

        subprocess.run(["fslswapdim",
                        input,
                        "-x", "y", "z",
                        output])


In [ ]:
# Mask Retardance by Cross
import os
from pathlib import Path
from fsl.wrappers import fslmaths#, LOAD

subject = '' #'Moe' #'Vlad'

inp_path = Path(f'/Users/Vasilis/CMC_data/{subject}/PSOCT')
target_mod  = 'Retardance'
mask_mod    = 'Cross'
out_path    = inp_path / (target_mod[:3] + '_masked_by_' + mask_mod[:3]) / 'lowres'
os.makedirs(out_path, exist_ok=True)

files = (inp_path / target_mod / 'lowres').glob('Slice_*_En*.nii.gz')
for f in files:
    mask_file = inp_path / mask_mod / 'lowres' / f.name.replace('EnR','EnCr')

    # mask = fslmaths(mask_file).bin().run(LOAD)

    fslmaths(f).mas(mask_file).run(out_path / f.name)


In [ ]:
# Process all PSOCT slides
from cmcmultimodal.core.proc import psoct
from pathlib import Path

datadir = Path('/Users/Vasilis/CMC_data/Vlad')
output_path = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409')
mri_ref = datadir / 'MRI/dti_FA.nii.gz'
seq_params = datadir / 'PSOCT/seq_params.json'

my_data = psoct(datadir, seq_params, psoct_reg_mod='Ret_masked_by_Cro', mri_reg_mod='Ret_masked_by_Cro', verbose=True)

my_data.run_pipeline(other_images=['Cross', 'Retardance', 'Reflectivity', 'Orientation'], 
                                    output_path=output_path, 
                                    mri_ref=mri_ref, 
                                    bad_slides=[],
                                    fnirt=True,
                                    invwarp=True)


In [ ]:
from pathlib import Path
from cmcmultimodal.utils.validate_results import compare_results_folder

ref_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret')
est_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret_new')

compare_results_folder(ref_path, est_path, subdir=True)

In [ ]:
import subprocess
inp_folder = out_file.parent / (modality[0:3] +'_temp_files')
files = sorted(inp_folder.glob('*.nii.gz'))
subprocess.run(['fslmaths',
                files[0],
                '-mul', '0',
                str(out_file)])
cmd = ['fslmaths', str(out_file)]
for i in files:
    cmd.extend(['-add', str(i)])
cmd.append(str(out_file))
subprocess.run(cmd)

In [29]:
from fsl.wrappers           import flirt

mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
outfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/dtifit_FA_in_Ret_PSOCT'
matfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/MRI_to_Ret_PSOCT.mat'

flirt(src=mri_ref, ref=out_file, out=outfile, omat=matfile, dof=12, interp='spline')


{}

In [11]:
import json
import numpy as np
from pathlib import Path

output_path = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Cross')

with open(output_path / "abs_mat.json") as f:
    data = json.load(f)

abs_mat = {int(k): np.array(v) for k, v in data.items()}